In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm
data = pd.read_csv("wrangled_data.csv")

The goal for this week is to complete the data wrangling part 2 portion of the project which focuses on encoding, derived features, date/time encoding, and numeric transformations. However, due to the nature of this project I will stray to some extent from the core goal. This is the week where I need to go through all the remaining articles in the data set and come up with a way to figure out if they are talking about short term or long term analysis and predictions and engineer/encode a new column that indicates this. Additionally, I need to finish the date/time encoding I started last week so yFinance can read the date and time and figure how how to handle days where the market is closed or how to standardized the time intervals and create the columns that have the tickers performances x amount of days after the article publication date. Lastly, we had a discussion in class about how skewed the data is in terms of articles published per author as discovered in the first data wrangling ipynb. The conclusion was that because any author can show up on the news regardless of how many articles they have written, I still need to classify those that have at least the minimum threshold of 25 articles so feature engineering a low, medium, high column for articles written can be helpful. Thinking really far ahead for just the dashboard, I might go back and assign a rating to all the authors that did not meet the 25 minimum article cutoff and give them all a poor rating because you should not really trust authors that have not published a lot or do not have a track record but you still need to know something about them because they can appear on your news feed. 

In [2]:
data.head()

,article_id,headline,url,publisher,published_at,ticker,Headline_Class
0,21991,"UBS Maintains Buy on Adobe, Raises Price Targe...",https://www.benzinga.com/news/20/06/16202690/u...,Benzinga Newsdesk,2020-06-08,ADBE,Analysis
1,21995,Shares of several technology companies are tra...,https://www.benzinga.com/wiim/20/05/16075931/s...,Benzinga Newsdesk,2020-05-20,ADBE,Analysis
2,21996,"Benzinga's Top Upgrades, Downgrades For May 14...",https://www.benzinga.com/markets/penny-stocks/...,Lisa Levin,2020-05-14,ADBE,Analysis
3,21997,"DZ Bank Downgrades Adobe to Hold, Announces $3...",https://www.benzinga.com/news/20/05/16029565/d...,Vick Meyer,2020-05-14,ADBE,Analysis
4,22000,"BMO Capital Maintains Outperform on Adobe, Rai...",https://www.benzinga.com/news/20/04/15921434/b...,Vick Meyer,2020-04-30,ADBE,Analysis


In [3]:
len(data)

28863

The steps im going to take is first research and learn how to do the scraping/classification for short term or long terms articles, add the yFinance columns, and then encode the article amounts.

Steps for scraping/classification: extract all the raw text out of the articles and ensure theres no errors or missing values, use navigator or equivalent to call a model like gpt 5 mini to evaluate the text, test 10-20 articles to make sure the outputs are valid, and then have it classify short term or long term on the entire dataset. 

Before I try to extract the text of 30,000 articles im going to test on a small subset of 50 articles and see if it is functioning.

In [4]:
test_text = data.sample(250, random_state=0).copy()

In [5]:
def get_text_from_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except Exception:
        return ""

tqdm.pandas()
test_text["article_text"] = test_text["url"].progress_apply(get_text_from_article)

100%|██████████| 250/250 [04:40<00:00,  1.12s/it]


In [6]:
test_text["article_text"]

20457    OMNOVA Solutions (NYSE:\n\n) is expected to re...
14403    Never miss a trade again with the fastest news...
11715    Never miss a trade again with the fastest news...
16180    Jon Najarian spoke on CNBC's "Fast Money Halft...
7081     Never miss a trade again with the fastest news...
                               ...                        
10211                                                     
12107                                                     
15609                                                     
8065                                                      
3089                                                      
Name: article_text, Length: 250, dtype: object

In [7]:
test_text["success"] = test_text["article_text"].str.len() > 50

In [90]:
success_rate = test_text["success"].mean()

print(f"Success rate: {success_rate:.2%}")
print(test_text[["url", "success"]])

Success rate: 50.00%
                                                     url  success
20457  https://www.benzinga.com/news/earnings/14/09/4...     True
14403  https://www.benzinga.com/markets/options/16/02...     True
11715  https://www.benzinga.com/markets/options/20/04...     True
16180  https://www.benzinga.com/markets/options/19/11...     True
7081   https://www.benzinga.com/news/16/03/7778820/sh...     True
22061  https://www.benzinga.com/analyst-ratings/analy...     True
2431   https://www.benzinga.com/analyst-ratings/price...     True
16092  https://www.benzinga.com/news/11/04/1031469/lo...     True
23079  https://www.benzinga.com/news/18/04/11515411/c...     True
6226   https://www.benzinga.com/analyst-ratings/upgra...     True
3368   https://www.benzinga.com/analyst-ratings/analy...    False
27286  https://www.benzinga.com/analyst-ratings/analy...    False
2604   https://www.benzinga.com/news/14/04/4478294/no...    False
2028   https://www.benzinga.com/fintech/17/03/9126577..

I ran the code once and it had about a 50% success rate and most of the remaining articles were behind a paywall but then I ran it again with the same random_state and it had a 0% success rate so I am going to have to investigate this further but its because I get detected as a bot for trying too much in a short amount of time.

In [91]:
data.head()

,article_id,headline,url,publisher,published_at,ticker,Headline_Class
0,21991,"UBS Maintains Buy on Adobe, Raises Price Targe...",https://www.benzinga.com/news/20/06/16202690/u...,Benzinga Newsdesk,2020-06-08,ADBE,Analysis
1,21995,Shares of several technology companies are tra...,https://www.benzinga.com/wiim/20/05/16075931/s...,Benzinga Newsdesk,2020-05-20,ADBE,Analysis
2,21996,"Benzinga's Top Upgrades, Downgrades For May 14...",https://www.benzinga.com/markets/penny-stocks/...,Lisa Levin,2020-05-14,ADBE,Analysis
3,21997,"DZ Bank Downgrades Adobe to Hold, Announces $3...",https://www.benzinga.com/news/20/05/16029565/d...,Vick Meyer,2020-05-14,ADBE,Analysis
4,22000,"BMO Capital Maintains Outperform on Adobe, Rai...",https://www.benzinga.com/news/20/04/15921434/b...,Vick Meyer,2020-04-30,ADBE,Analysis


After a lot of reading and reflection I want to discuss this in class but I think that because I am going to be evaluating publishers based on alignment between sentiment and stock performance, the return windows of 7 days, 30 days, and 6 months already define the time spans. This means that I will already have 3 models to evaluate performance, one for each time interval so its implicit the authors focus for articles or their performance. This will let me analyze their performance on multiple time scales without relying on any scraping with missing data and still provide valuable information to readers. I think it would be interesting to know oh this publisher has a really good score for short term discussion but falls off on the long term.

For feature engineering this week I am going to drop article_id because that gives no predictive value, ticker will not be used in the final model buts its needed for yFinance data so im keeping it for now, Headline_Class will be one hot encoded, I will create the column we talked about for the low, medium, high encoded article counts, and the 3 yFinance return columns.

In [92]:
data = data.drop('article_id', axis=1)
data.head()

,headline,url,publisher,published_at,ticker,Headline_Class
0,"UBS Maintains Buy on Adobe, Raises Price Targe...",https://www.benzinga.com/news/20/06/16202690/u...,Benzinga Newsdesk,2020-06-08,ADBE,Analysis
1,Shares of several technology companies are tra...,https://www.benzinga.com/wiim/20/05/16075931/s...,Benzinga Newsdesk,2020-05-20,ADBE,Analysis
2,"Benzinga's Top Upgrades, Downgrades For May 14...",https://www.benzinga.com/markets/penny-stocks/...,Lisa Levin,2020-05-14,ADBE,Analysis
3,"DZ Bank Downgrades Adobe to Hold, Announces $3...",https://www.benzinga.com/news/20/05/16029565/d...,Vick Meyer,2020-05-14,ADBE,Analysis
4,"BMO Capital Maintains Outperform on Adobe, Rai...",https://www.benzinga.com/news/20/04/15921434/b...,Vick Meyer,2020-04-30,ADBE,Analysis


In [93]:
data = pd.get_dummies(data, columns=['Headline_Class'], drop_first=True, dtype=int)
data.head()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction
0,"UBS Maintains Buy on Adobe, Raises Price Targe...",https://www.benzinga.com/news/20/06/16202690/u...,Benzinga Newsdesk,2020-06-08,ADBE,0
1,Shares of several technology companies are tra...,https://www.benzinga.com/wiim/20/05/16075931/s...,Benzinga Newsdesk,2020-05-20,ADBE,0
2,"Benzinga's Top Upgrades, Downgrades For May 14...",https://www.benzinga.com/markets/penny-stocks/...,Lisa Levin,2020-05-14,ADBE,0
3,"DZ Bank Downgrades Adobe to Hold, Announces $3...",https://www.benzinga.com/news/20/05/16029565/d...,Vick Meyer,2020-05-14,ADBE,0
4,"BMO Capital Maintains Outperform on Adobe, Rai...",https://www.benzinga.com/news/20/04/15921434/b...,Vick Meyer,2020-04-30,ADBE,0


In [94]:
publisher_count = data['publisher'].value_counts()
publisher_count

publisher
Paul Quintaro        4244
Charles Gross        2889
Benzinga Newsdesk    2428
Lisa Levin           2321
Vick Meyer           1968
                     ... 
Alex Furno             29
John Harris            29
Spencer Israel         27
Luke J Jacobi          25
Hilary Farrell         25
Name: count, Length: 86, dtype: int64

In [95]:
data['total_article_count'] = data['publisher'].map(publisher_count)
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count
28858,"Wedbush Initiates Zynga at Outperform, $12.50 PT",https://www.benzinga.com/analyst-ratings/price...,Juan Lopez,2011-12-20,ZNGA,0,1314
28859,"Wedbush Initiates Zynga Coverage: Outperform, ...",https://www.benzinga.com/analyst-ratings/analy...,Hilary Farrell,2011-12-19,ZNGA,0,25
28860,BTIG Initiates Coverage of Zynga with Buy Rati...,https://www.benzinga.com/analyst-ratings/initi...,Matthew Kennedy,2011-12-16,ZNGA,0,76
28861,"UPDATE: Sterne Agee Initiates Underperform, PT...",https://www.benzinga.com/analyst-ratings/analy...,David Johnson,2011-12-13,ZNGA,0,378
28862,"Sterne Agee Initiates Zynga at Underperform, $...",https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-13,ZNGA,0,401


In [96]:
data['total_article_count'].describe()

count    28863.000000
mean      1738.361536
std       1390.769282
min         25.000000
25%        413.000000
50%       1314.000000
75%       2428.000000
max       4244.000000
Name: total_article_count, dtype: float64

In [97]:
def encode_article_counts(publisher):
    if publisher <= 413:
        return 0
    elif publisher <= 2428:
        return 1
    else:
        return 2
data['publisher_size'] = data['total_article_count'].apply(encode_article_counts)
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size
28858,"Wedbush Initiates Zynga at Outperform, $12.50 PT",https://www.benzinga.com/analyst-ratings/price...,Juan Lopez,2011-12-20,ZNGA,0,1314,1
28859,"Wedbush Initiates Zynga Coverage: Outperform, ...",https://www.benzinga.com/analyst-ratings/analy...,Hilary Farrell,2011-12-19,ZNGA,0,25,0
28860,BTIG Initiates Coverage of Zynga with Buy Rati...,https://www.benzinga.com/analyst-ratings/initi...,Matthew Kennedy,2011-12-16,ZNGA,0,76,0
28861,"UPDATE: Sterne Agee Initiates Underperform, PT...",https://www.benzinga.com/analyst-ratings/analy...,David Johnson,2011-12-13,ZNGA,0,378,0
28862,"Sterne Agee Initiates Zynga at Underperform, $...",https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-13,ZNGA,0,401,0


Now the hard part is actually adding the yFinance columns. I need a formula to ensure consistency across all the data. For the starting date by default it is the published date but if that is not on a trading day then the next possible trading day becomes the starting point. Same idea for the end of the interval, the first valid trading day on or after the 7 day, 30 day, 6 month period.

In [98]:
data.dtypes

headline                     object
url                          object
publisher                    object
published_at                 object
ticker                       object
Headline_Class_Prediction     int64
total_article_count           int64
publisher_size                int64
dtype: object

In [99]:
data['published_at'].head()

0    2020-06-08
1    2020-05-20
2    2020-05-14
3    2020-05-14
4    2020-04-30
Name: published_at, dtype: object

In [100]:
data['published_at'] = pd.to_datetime(data['published_at'], errors='raise').dt.normalize()

In [101]:
data.dtypes

headline                             object
url                                  object
publisher                            object
published_at                 datetime64[ns]
ticker                               object
Headline_Class_Prediction             int64
total_article_count                   int64
publisher_size                        int64
dtype: object

In [102]:
import yfinance as yf
columns = {
    "return_7d": 7,
    "return_30d": 30,
    "return_6m": 182
}
download_date = (data['published_at'].min() - pd.Timedelta(days=7)).date()
download_end = (data['published_at'].max() + pd.Timedelta(days=182)).date()
current_tickers = data['ticker'].unique()

In [103]:
current_tickers

array(['ADBE', 'AIG', 'AMD', 'ATVI', 'AVGO', 'AXP', 'AZN', 'BABA', 'BAC',
       'BIIB', 'BMY', 'CAT', 'CMCSA', 'CMG', 'COP', 'DB', 'DE', 'DG',
       'EA', 'EBAY', 'FDX', 'GILD', 'GME', 'HAL', 'HD', 'INTC', 'JNJ',
       'KSS', 'LLY', 'LMT', 'LOW', 'MA', 'MCD', 'MDT', 'MRK', 'MS', 'MU',
       'MYL', 'NFLX', 'NOK', 'NVDA', 'ORCL', 'PFE', 'QCOM', 'TSLA', 'TXN',
       'WFC', 'YELP', 'YUM', 'ZNGA'], dtype=object)

In [104]:
prices = {}
delisted_tickers = []
for ticker in current_tickers:
    ydata = yf.download(ticker, start=download_date, end=download_end, progress=False)
    if ydata is None or ydata.empty:
        delisted_tickers.append(ticker)
    # if the tickers delisted keep track so I can remove them later
    # otherwise convert to proper date format, fix the double array and check both adj close and close because data is different
    ydata.index = pd.to_datetime(ydata.index).normalize()
    if isinstance(ydata.columns, pd.MultiIndex):
        ydata = ydata.xs(ticker, axis=1, level=1, drop_level=True) # I looked up how to do this step yfinance does not have consistent formatting for some reason
    if 'Adj Close' in ydata.columns:
        prices[ticker] = ydata['Adj Close'].dropna()
    elif 'Close' in ydata.columns:
        prices[ticker] = ydata['Close'].dropna()
    else:
        delisted_tickers.append(ticker)
#print(ydata.columns)
#print(ydata.head())
print("Tickers:", len(prices))
print("Delisted:", len(delisted_tickers))
print(delisted_tickers)

$ATVI: possibly delisted; no timezone found

1 Failed download:
['ATVI']: possibly delisted; no timezone found
$MYL: possibly delisted; no timezone found

1 Failed download:
['MYL']: possibly delisted; no timezone found
$ZNGA: possibly delisted; no timezone found

1 Failed download:
['ZNGA']: possibly delisted; no timezone found


Tickers: 50
Delisted: 3
['ATVI', 'MYL', 'ZNGA']


In [105]:
del_tickers = set(delisted_tickers)
num_rows = data['ticker'].isin(del_tickers).sum()
print(num_rows)

1460


1460 rows that use delisted tickers. Since its a small amount of data im just going to remove them, remove any publishers that dropped below my 25 threshold and redo the encoding because the mean and std are going to change. 

In [106]:
row_removal = data[data['ticker'].isin(delisted_tickers)].index
data = data.drop(row_removal).copy()
len(data)

27403

In [107]:
publisher_count = data['publisher'].value_counts()
publisher_count.describe()

count      86.000000
mean      318.639535
std       652.997089
min        23.000000
25%        39.250000
50%        72.000000
75%       261.750000
max      4043.000000
Name: count, dtype: float64

In [108]:
publisher_amount = data['publisher'].value_counts()
publisher_cutoff = publisher_amount[publisher_amount >= 25].index
data_filtered_publisher = data[data['publisher'].isin(publisher_cutoff)].copy()
len(data_filtered_publisher)

27357

In [109]:
data_filtered_publisher['total_article_count'].describe()

count    27357.000000
mean      1736.017582
std       1390.166651
min         25.000000
25%        413.000000
50%       1314.000000
75%       2428.000000
max       4244.000000
Name: total_article_count, dtype: float64

Only 50 rows were removed so 25, 50, 75% cutoffs did not change so no need to redo encoding

Now I need to make the 3 yfinance columns, transfer the data and check everything worked

In [110]:
data = data_filtered_publisher.copy()

In [111]:
data['return_7d'] = np.nan
data['return_30d'] = np.nan
data['return_6m'] = np.nan
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m
28458,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,NaN,NaN,NaN
28459,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,NaN,NaN,NaN
28460,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,NaN,NaN,NaN
28461,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,NaN,NaN,NaN
28462,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,NaN,NaN,NaN


In [112]:
for i in range(len(data)):
    # go through each row
    ticker = data.iloc[i]['ticker'] # get the ticker at the row
    date = data.iloc[i]['published_at'] # get the date article was published for the ticker at the row

    price_history = prices[ticker] # get the entire price history for the current ticker of the current row
    trading_days = price_history.index[price_history.index >= date] #get all valid trading days on or after published at date ie no holiday no weekend

    first_valid_day = trading_days[0] # first trading day on or after publication date
    first_stock_price = float(price_history.loc[first_valid_day]) # get stock price on first trading day on or after publicationd ate

    # now handle 7 day, 30, 6m returns 7day first

    seven_day = first_valid_day + pd.Timedelta(days=7) # get 7 days after published date then need to check if trading day
    valid_seven_days = price_history.index[price_history.index >= seven_day] #first valid trading days oon or after the 7th day

    if (len(valid_seven_days) > 0):
        valid_seven_day = valid_seven_days[0]
        seven_day_price = float(price_history.loc[valid_seven_day]) # stock price in the 7 day

        data.iloc[i, data.columns.get_loc('return_7d')] = (seven_day_price - first_stock_price) / first_stock_price # store the % return in 7 days

    # same thing for 30 day and 6 month

    thirty_day = first_valid_day+ pd.Timedelta(days=30) # get 30 days after published date then need to check if trading day
    valid_thirty_days = price_history.index[price_history.index >= thirty_day] #first valid trading days oon or after the 30th day

    if (len(valid_thirty_days) > 0):
        valid_thirty_day = valid_thirty_days[0]
        thirty_day_price = float(price_history.loc[valid_thirty_day]) # stock price in the 30 day

        data.iloc[i, data.columns.get_loc('return_30d')] = (thirty_day_price - first_stock_price) / first_stock_price # store the % return in 30 days


    six_month = first_valid_day + pd.Timedelta(days=182) # get 182 days after published date then need to check if trading day
    valid_six_months = price_history.index[price_history.index >= six_month] #first valid trading days oon or after the 182nd day

    if (len(valid_six_months) > 0):
        valid_six_month = valid_six_months[0]
        six_month_price = float(price_history.loc[valid_six_month]) # stock price in the 6th month

        data.iloc[i, data.columns.get_loc('return_6m')] = (six_month_price - first_stock_price) / first_stock_price # store the % return in 182 days



In [113]:
data[['return_7d', 'return_30d', 'return_6m']].describe()

,return_7d,return_30d,return_6m
count,27357.000000,27357.000000,27343.000000
mean,0.003196,0.015666,0.127252
std,0.053642,0.106197,0.335118
min,-0.457053,-0.673156,-0.730762
25%,-0.019812,-0.035492,-0.040199
50%,0.003393,0.015902,0.082839
75%,0.025773,0.063101,0.220809
max,0.492858,1.087066,5.114833


In [115]:
data[['return_7d', 'return_30d', 'return_6m']].isna().mean()

return_7d     0.000000
return_30d    0.000000
return_6m     0.000512
dtype: float64

In [116]:
data.tail(20)

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m
28443,Morgan Stanley Maintains Yum! Brands at Overwe...,https://www.benzinga.com/analyst-ratings/price...,webmaster,2012-04-16,YUM,0,354,0,0.021176,-0.029204,-0.010321
28444,"UPDATE: Oppenheimer Initiates Outperform, $82 ...",https://www.benzinga.com/analyst-ratings/analy...,David Johnson,2012-03-19,YUM,0,378,0,0.034462,0.060597,-0.028070
28445,Oppenheimer Initiates Coverage on Yum! Brands ...,https://www.benzinga.com/analyst-ratings/price...,webmaster,2012-03-19,YUM,0,354,0,0.034462,0.060597,-0.028070
28446,UPDATE: Credit Suisse Raises PT to $73 on Yum ...,https://www.benzinga.com/analyst-ratings/analy...,David Johnson,2012-02-09,YUM,0,378,0,-0.002773,0.031120,0.035158
28447,UPDATE: Deutsche Bank Raises Target to $66 on ...,https://www.benzinga.com/analyst-ratings/analy...,David Johnson,2012-02-09,YUM,0,378,0,-0.002773,0.031120,0.035158
28448,UPDATE: Jefferies Raises PT to $60 on YUM! Brands,https://www.benzinga.com/analyst-ratings/analy...,David Johnson,2012-02-08,YUM,0,378,0,-0.011173,0.046089,0.042239
28449,Goldman Sachs Upgrades Yum! Brands from Sell t...,https://www.benzinga.com/analyst-ratings/upgra...,webmaster,2012-02-08,YUM,0,354,0,-0.011173,0.046089,0.042239
28450,Jefferies &amp; Company Maintains Yum! Brands ...,https://www.benzinga.com/analyst-ratings/price...,webmaster,2012-02-08,YUM,0,354,0,-0.011173,0.046089,0.042239
28451,Yum Expects China Same Store Sales to Moderate...,https://www.benzinga.com/news/earnings/12/02/2...,Matthew Kennedy,2012-02-07,YUM,1,76,0,-0.013416,0.032999,0.043893
28452,Yum Brands Says Half of Operating Profit is fr...,https://www.benzinga.com/news/earnings/12/02/2...,Matthew Kennedy,2012-02-07,YUM,0,76,0,-0.013416,0.032999,0.043893


In [119]:
data.to_csv('data_wrangling_part2.csv', index=False)